# Papal visits: does the Vatican pursue a geopolitical objective?

Following Pope Leo's visit to France in Fall 2026 and given the media coverage it received, we decided to investigate the Vatican's diplomacy and to determine whether it pursues a geopolitical goal. To do this, we first analysed the countries visited by John-Paul II, Benedict XVI and Francis. We cross the list of countries visited by these popes with the religious composition of these countries and their level of development.

Questions:

1. **Who do the popes speak to?** Are the visited countries mostly Catholic, or at least more Catholic than the world average?
2. **Growth or decline?** Do the popes go where Catholicism is growing or where it is declining?
3. **Development**: do they rather go to rich countries or to poor countries?

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import s3fs
from matplotlib.patches import Patch

# The files are in the public "diffusion" folder: anonymous access works for everyone, without SSP Cloud credentials
fs = s3fs.S3FileSystem(anon=True, client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})

POPES = ["John-Paul II", "Benedict XVI", "Francis"]
# Start and end of each pontificate
PONTIFICATES = {"John-Paul II": (1978, 2005), "Benedict XVI": (2005, 2013), "Francis": (2013, 2025)}

### The trips data : webscrapping the Vatican website

As there isn't any source listing the Popes' visits, we resorted to webscrapping and got the list of the trips from the vatican website directly. **We complied with the document 'robots.txt' and implemented a two seconds lap between two queries**. The result has then been cleaned and saved on the s3 storage system. The scripts used are scrapping_no_speeches.py and cleaning_no_speeches (the version with spacy in archives_cleaning_spacy was not efficient enough and AI has been used to resolve the issues in the actual script). For the analysis, we directly use the files from the s3 storage.

In [ ]:
with fs.open("avibert/diffusion/df_trips_countries.parquet", "rb") as f:
    trips = pd.read_parquet(f)

trips = trips.rename(columns={"Pope": "pope", "Year": "year", "Place": "trip",
                              "Country": "country", "ISO3": "iso3"})
trips["year"] = trips["year"].astype(int)
trips.head(3)

### The religions: RCS-Dem 2.0

Our first step to enrich the dataset was to combine it with the estimated share of every religion in every country visited. We used the RCS-Dem 2.0 dataset from David Brown and Patrick James and uploaded on s3. The dataset is from 2019. Its structure is one row per country and per year (from 1900 to 2015). The columns ending in `PC` are shares of the population (in %), `TOTPP` is the total population. We keep the main religions and the detail of Christians. The rest of the population that is not included in any category (Chinese folk religions, African religions, syncretic Christians...) is put in an `other_religions` column, except the people whose religion is unknown (`UNKPC`). We decided to keep both Christian and its subdivision to refine the analysis. 

In [ ]:
FILE_RCS = ("avibert/diffusion/Religious Characteristics of States Dataset Project - "
            "Demographics v. 2.0 (RCS-Dem 2.0), COUNTRIES ONLY.SAV")
with fs.open(FILE_RCS, "rb") as f:
    rcs_raw = pd.read_spss(f)

# We give West-Germany the iso3 code of Germany
rcs_raw.loc[rcs_raw["ABBREV"] == "GFR", "ISO3"] = "DEU"

# We make the name of the columns more explicit
religions = pd.DataFrame({
    "iso3": rcs_raw["ISO3"].astype(str).str.strip(),
    "year": rcs_raw["YEAR"],
    "population": rcs_raw["TOTPP"],
    "christians": rcs_raw["CHRPC"],
    "muslims": rcs_raw["MUSPC"],
    "jews": rcs_raw["JEWPC"],
    "buddhists": rcs_raw["BUDPC"],
    "hindus": rcs_raw["HINPC"],
    "no_religion": rcs_raw["NREPC"],
    "catholics": rcs_raw["CATPC"],
    "protestants": rcs_raw["PRTPC"],
    "orthodox": rcs_raw["ORTPC"],
    "unknown": rcs_raw["UNKPC"],
})

# We create the "other religion" column
main_religions = ["christians", "muslims", "jews", "buddhists", "hindus", "no_religion"]
religions["other_religions"] = 100 - religions[main_religions].sum(axis=1) - religions["unknown"].fillna(0)
# Small negative values come from rounding in the data: we put them to 0
religions["other_religions"] = religions["other_religions"].clip(lower=0)
# If a country has no data on religions, the other religions are unknown too
religions.loc[religions["christians"].isna(), "other_religions"] = np.nan
# Some old states have no ISO3 code: we remove them
religions = religions[religions["iso3"] != ""]
religions = religions.dropna(subset=["year"])
religions["year"] = religions["year"].astype(int)
religions.describe()

### GDP per capita, region and income (World Bank)

The second step to enrich the data was to add information on the economic development of the visited countries. GDP per capita in constant 2015 dollars (indicator `NY.GDP.PCAP.KD`) has been retrieved from the World Bank, downloaded from https://data.worldbank.org/indicator/NY.GDP.PCAP.KD (CSV).

In [ ]:
FILE_GDP = "avibert/diffusion/API_NY.GDP.PCAP.KD_DS2_en_csv_v2_71952.csv"
with fs.open(FILE_GDP, "rb") as f:
    gdp_raw = pd.read_csv(f, skiprows=4)   # the first 4 lines are a title, not the table

# The last column is empty (the lines end with a comma): we remove the empty columns
gdp_raw = gdp_raw.dropna(axis=1, how="all")

# The data is in a wide format, we convert it with melt in a long format
wb = gdp_raw.drop(columns=["Country Name", "Indicator Name", "Indicator Code"])
wb = wb.melt(id_vars="Country Code", var_name="year", value_name="gdp_per_capita")
wb = wb.rename(columns={"Country Code": "iso3"})
wb["year"] = wb["year"].astype(int)
wb.head()

In [ ]:
# Region and income group of each country (current classification)
FILE_COUNTRIES = "avibert/diffusion/Metadata_Country_API_NY.GDP.PCAP.KD_DS2_en_csv_v2_71952.csv"
with fs.open(FILE_COUNTRIES, "rb") as f:
    wb_countries = pd.read_csv(f)

wb_countries = wb_countries[["Country Code", "Region", "IncomeGroup"]]
wb_countries = wb_countries.rename(columns={"Country Code": "iso3", "Region": "region", "IncomeGroup": "income"})
# The aggregates (World, Euro area...) have no region: we remove them
wb_countries = wb_countries.dropna(subset=["region"])
wb_countries.head(3)

### Join the trips with the data of the closest year

RCS-Dem stops in 2015. For each visit, we take the closest observation (`merge_asof`). We keep the gap in years to control it later on and make sure we do not use irrelevant data in our analysis. For the visits after 2015 (Francis), we assume that the share did not change drastically between 2015 and 2024 use the 2015 data.

In [ ]:
religions = religions.rename(columns={"year": "year_religion"}).sort_values("year_religion")

visits = pd.merge_asof(
    trips.sort_values("year"),
    religions,
    left_on="year", right_on="year_religion", by="iso3",
    direction="nearest")
visits["gap_religion"] = visits["year_religion"] - visits["year"]

In [ ]:
# GDP: closest year, at most 5 years of difference
wb_sorted = wb.rename(columns={"year": "year_wb"}).sort_values("year_wb")
visits = pd.merge_asof(visits.sort_values("year"), wb_sorted,
                       left_on="year", right_on="year_wb", by="iso3",
                       direction="nearest", tolerance=5)
visits = visits.merge(wb_countries, on="iso3", how="left")

# Majority religion: we only compare the main religions and the other religions
majority_columns = main_religions + ["other_religions"]
# We drop the na lines to compute the majority for the countries that have data only
visits["majority"] = visits[majority_columns].dropna(how="all").idxmax(axis=1)

### Check the join

In [ ]:
# For each pope: how many visits without data, and what is the average gap in years?
visits.groupby("pope").agg(
    nb_visits=("iso3", "size"),
    no_religion_data=("christians", lambda x: x.isna().sum()),
    no_catholic_data=("catholics", lambda x: x.isna().sum()),
    mean_gap_religion=("gap_religion", lambda x: x.abs().mean()),
).round(1)

In [ ]:
# Visited countries where the other religions are the majority
visits.loc[visits["majority"] == "other_religions", ["pope", "year", "country", "christians", "other_religions"]].round(1).head(3)

All the instructions went well and we do not see issues in the joins

## 2. Overview of the trips

The first step of our analysis is to get an overall view of the trips. We aim to answer the question : **Were trips equally important for every pope and were every continent equally important for them ?** By comparing the different popes, we might find in the discrepancies signs that visits pursue specific goals other than mere pastoral visits.

In [ ]:
# One trip = one trip title for a pope and a year
trips["trip_id"] = trips["pope"] + " " + trips["year"].astype(str) + " " + trips["trip"]

overview = trips.groupby("pope").agg(
    nb_trips=("trip_id", "nunique"),
    nb_visits=("iso3", "size"),
    nb_countries=("iso3", "nunique"),
).loc[POPES]
overview["pontificate_length"] = [end - start for start, end in PONTIFICATES.values()]
overview["trips_per_year"] = (overview["nb_trips"] / overview["pontificate_length"]).round(1)
overview

In [ ]:
per_year = trips.groupby(["year", "pope"])["trip_id"].nunique().unstack(fill_value=0)[POPES]
per_year.plot.bar(stacked=True, figsize=(12, 4), width=0.8)
plt.ylabel("Number of trips")
plt.title("Trips outside Italy each year")
plt.show()

John-Paul II and Francis have been more active than Benedict XVI with 4 trips per year on average for the first two against 3 for the latter. John-Paul II took advantage of his youth since he was elected at only 58. Francis, who was almost the same age as Benedict XVI, shows a strong wilingness to visit countries bespite his age. He traveled more than John-Paul II at the same age.

In [ ]:
# The most visited countries
trips.groupby("country")["trip_id"].nunique().sort_values(ascending=False).head(15)

In [ ]:
# Does John-Paul II accounts for all the Poland trips ?

mask = (trips["pope"]== "John-Paul II") & (trips["country"] == "Poland")
mask.sum()

The countries that are the most visited are either European or North American. This would imply, as these countries are mostly christian, that the Vatican opts more often for countries which have or used to have a predominently christian population. We were surprised by Poland's figure but the peculiarity disappears when we see that John-Paul II, born in Poland, accounts for 9 trips of the 11 in total. 

### Maps of the visited countries

By representing the data on maps, we can visually see if some regions have been left aside and compare the Popes' choices more easily.

In [ ]:
world = gpd.read_file("https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip")
world = world[["NAME", "ISO_A3_EH", "geometry"]].rename(columns={"NAME": "name", "ISO_A3_EH": "iso3"})
world = world[world["name"] != "Antarctica"]

In [ ]:
# Map 1: number of popes (among the three) who visited each country
nb_popes = trips.groupby("iso3")["pope"].nunique().reset_index(name="nb_popes")
world_popes = world.merge(nb_popes, on="iso3", how="left")
world_popes["nb_popes"] = world_popes["nb_popes"].fillna(0)

colors = {1: "#9ecae1", 2: "#4292c6", 3: "#08306b"}

fig, ax = plt.subplots(figsize=(14, 7))
world_popes.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.3)
for n, color in colors.items():
    world_popes[world_popes["nb_popes"] == n].plot(ax=ax, color=color, edgecolor="white", linewidth=0.3)

legend = [Patch(color="lightgrey", label="not visited")]
for n, color in colors.items():
    legend.append(Patch(color=color, label=f"visited by {n} pope(s)"))
ax.legend(handles=legend, loc="lower left")
ax.set_title("Countries visited by John Paul II, Benedict XVI and Francis")
ax.set_axis_off()
plt.show()

In [ ]:
# Maps 2 to 4: the countries visited by each pope
fig, axes = plt.subplots(3, 1, figsize=(12, 18))
for ax, pope in zip(axes, POPES):
    visited_iso3 = trips.loc[trips["pope"] == pope, "iso3"]
    world.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.3)
    world[world["iso3"].isin(visited_iso3)].plot(ax=ax, color="#2171b5", edgecolor="white", linewidth=0.3)
    ax.set_title(f"Countries visited by {pope}")
    ax.set_axis_off()
plt.show()

The maps show large differences between popes. Together, they have visited most countries except Russia and Chine (the largest countries left unvisited), the horn of Africa and Central Asia. John-Paul II roamed almost all of Europe and America, and most African countries. Benedict XVI focused on Europe and key countries in America (US, Mexico and Brazil). The comparison between Francis and John-Paul II is interesting. Both visited numerous countries in America but Francis picked mostly Eastern European and Asian countries when his predecessor visited all of Europe indifferently and Africa.

## 3. Who do the popes speak to?

To compare the visited countries with the others, we build a table with **one row per pope and per country of the world**, and a column `visited` which is True if the pope went there. Each country is counted once, even if the pope went there several times.

We use the religious data of the middle year of the pontificate (2015 for Francis, because RCS-Dem stops in 2015).

In [ ]:
rows = []
for pope, (start, end) in PONTIFICATES.items():
    middle_year = min((start + end) // 2, 2015)
    table = religions[religions["year_religion"] == middle_year].copy()
    table["pope"] = pope
    visited_iso3 = trips.loc[trips["pope"] == pope, "iso3"]
    table["visited"] = table["iso3"].isin(visited_iso3)
    rows.append(table)

countries = pd.concat(rows)
countries["majority"] = countries[majority_columns].dropna(how="all").idxmax(axis=1)

# Number of visited countries and number of countries for each pope
countries.groupby("pope")["visited"].agg(["sum", "size"]).loc[POPES]

The table is consistent with the previous results. There is a difference in size because Tchecoslovakia was still an entry in 1991 and the Dutch West Indies in 2009.

### Share of Catholics in the visited countries

In [ ]:
visited_countries = countries[countries["visited"]]
visited_countries.groupby("pope")["catholics"].describe().loc[POPES].round(1)

In [ ]:
# Same thing for the countries that were not visited, to compare
countries[~countries["visited"]].groupby("pope")["catholics"].describe().loc[POPES].round(1)

The tables confirm the hypothesis made earlier that the Popes were opting more often for catholic countries than others. The mean share of catholics in visited countries is significantly higher than in the other countries. One issue is that the figure does not account alone for countries religiously diverse where Catholicism could be the predominant religion without representing a high share of the population. Hence the following analysis.

### How many visited countries are Christian or Catholic?

- Christian: Christians are the largest religion of the country (`majority` column).
- Catholic: Catholics are the largest group of the country when Christians are split into Catholics, Protestants and Orthodox and compared with the other religions (the `christians` column is left aside).

In [ ]:
visited_countries = countries[countries["visited"]].copy()
visited_countries["christian_majority"] = visited_countries["majority"] == "christians"

# Largest group when the Christians are split into Catholics, Protestants and Orthodox
detail_columns = ["catholics", "protestants", "orthodox"] + [c for c in majority_columns if c != "christians"]
largest_group = visited_countries[detail_columns].dropna(how="all").idxmax(axis=1)
visited_countries["catholic_majority"] = largest_group.reindex(visited_countries.index) == "catholics"

result = visited_countries.groupby("pope")[["christian_majority", "catholic_majority"]].sum()
result["nb_visited_countries"] = visited_countries.groupby("pope").size()
result["% christian_majority"] = (result["christian_majority"] / result["nb_visited_countries"] * 100).round(1)
result["% catholic_majority"] = (result["catholic_majority"] / result["nb_visited_countries"] * 100).round(1)
result.loc[POPES]

The table confirms that Popes are more likely to visit countries with a christian and in particular a catholic majority. The large percentages for christian majority is not completely surprising has protestant countries such the US or Germany also have an important catholic population, albeit in minority. 

### Distribution of the share of Catholics

The means can hide very different situations. We compare the whole distribution of the share of Catholics in the visited and in the non-visited countries (each country counts once; the histograms are normalised so that both groups can be compared despite their different sizes).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, pope in zip(axes, POPES):
    table = countries[countries["pope"] == pope]
    ax.hist(table.loc[~table["visited"], "catholics"].dropna(), bins=20, alpha=0.5, label="not visited", density=True)
    ax.hist(table.loc[table["visited"], "catholics"].dropna(), bins=20, alpha=0.5, label="visited", density=True)
    ax.set_title(pope)
    ax.set_xlabel("Share of Catholics (%)")
axes[0].legend()
plt.show()

The non-visited countries are concentrated near 0 %: most countries of the world have almost no Catholics. The three popes differ in how far they move away from this pattern:

- **John-Paul II**: the visited countries are much less often at 0 % than the non-visited ones, and there is a clear bump between 75 % and 90 %. He went to almost every very Catholic country, but he also visited many countries with few Catholics, which is consistent with the number of countries he covered.
- **Benedict XVI**: the countries with almost no Catholics are rare among his visits. His visits are spread between 20 % and 90 %, which matches his focus on Europe and on large Catholic countries of America.
- **Francis**: the two distributions are very close. The visited countries are as often near 0 % as the non-visited ones (Asia, Middle East), and the bump around 80 % is smaller. The difference of means for Francis therefore relies on a limited number of very Catholic countries, and his trips are the least oriented towards Catholic countries of the three.

## 4. How Christian population evolves in visited countries ?

To go further in the analysis, we thought that looking at the dynamics of the catholic population at the time of the visit could also be interesting. The goal is to determine whether the Vatican has a defensive or an offensive diplomacy. Does it targets countries where the share of catholics increases or where it decreases to attempt halting its demise ?

For each visit, we compute the change of the share of Catholics in the visited country during the **5 years before the visit** (in percentage points). We compare it with the change of the **average share of Catholics of all countries** during the same 5 years (each country counts for one, like in part 3).

RCS-Dem stops in 2015: for the visits after 2015, we use the change between 2010 and 2015.

In [ ]:
# Average share of Catholics of all countries, each year
mean_catholics = religions.groupby("year_religion")["catholics"].mean().reset_index(name="mean_catholics")

# The first visit is in 1979, so the first year we need is 1974
mean_catholics[mean_catholics["year_religion"] >= 1974].plot(x="year_religion", y="mean_catholics", legend=False)
plt.xlabel("Year")
plt.ylabel("Share of Catholics (%)")
plt.title("Average share of Catholics of all countries")
plt.show()

We see that the share of catholics is declining in general. Consequently, we expect the pope to visit countries where the catholic population is also receding. A different result would imply an active diplomacy targeting specific countries.

In [ ]:
# The 5 years before the visit (the data stops in 2015)
visits["year_end"] = visits["year"].clip(upper=2015)
visits["year_start"] = visits["year_end"] - 5

# Share of Catholics in the visited country at the start and at the end
share_start = religions[["iso3", "year_religion", "catholics"]].rename(
    columns={"year_religion": "year_start", "catholics": "catholics_start"})
share_end = religions[["iso3", "year_religion", "catholics"]].rename(
    columns={"year_religion": "year_end", "catholics": "catholics_end"})
visits = visits.merge(share_start, on=["iso3", "year_start"], how="left")
visits = visits.merge(share_end, on=["iso3", "year_end"], how="left")

# Average share of Catholics of all countries at the start and at the end
mean_start = mean_catholics.rename(columns={"year_religion": "year_start", "mean_catholics": "mean_start"})
mean_end = mean_catholics.rename(columns={"year_religion": "year_end", "mean_catholics": "mean_end"})
visits = visits.merge(mean_start, on="year_start", how="left")
visits = visits.merge(mean_end, on="year_end", how="left")

# Changes in percentage points
visits["change_country"] = visits["catholics_end"] - visits["catholics_start"]
visits["change_mean"] = visits["mean_end"] - visits["mean_start"]
visits["change_vs_mean"] = visits["change_country"] - visits["change_mean"]

### Descriptive statistics

In [ ]:
changes = visits.dropna(subset=["change_country"]).copy()
changes.groupby("pope")["change_country"].describe().loc[POPES].round(2)

In [ ]:
# Average change in the visited countries and in all countries
changes.groupby("pope")[["change_country", "change_mean", "change_vs_mean"]].mean().loc[POPES].round(2)

In [ ]:
# Share of visits where Catholicism was growing, and growing faster than on average
changes["growing"] = changes["change_country"] > 0
changes["faster_than_mean"] = changes["change_vs_mean"] > 0

(changes.groupby("pope")[["growing", "faster_than_mean"]].mean() * 100).loc[POPES].round(1)

In [ ]:
changes.boxplot(column="change_vs_mean", by="pope")
plt.axhline(0, color="grey")
plt.title("Change of the share of Catholics, compared with the average of all countries (5 years before the visit)")
plt.suptitle("")
plt.xlabel("")
plt.ylabel("Percentage points")
plt.show()

The data is very enlighting. We see a great difference between Francis and Benedict XVI. Benedict visited countries that were losing catholics at a faster pace than the mean. Om the opposite, the countries which were visited by Francis show an average evolution rate of almost 0%. This shows a clear tropism toward countries where catholicism is expanding. We can therefore argue that Francis had an "offensive diplomacy" whereas Benedict XVI was more defensive. John-Paul II occupies the middle ground. 

To complete the analysis, a further analysis of the countries themself and the evolution of their population in terms of religion would help. For example, we could look for pope Francis whether the countries with few catholics are the one with the greatest evolution or not.

## 5. Development: do the popes go to rich or poor countries?

Last analysis before getting to the speeches, we wanted to look at the visited countries' level of development. This would supplement the analysis of the targeted population in terms of religion with an economic perspective.

For each visit, we divide the GDP per capita of the visited country by the world GDP per capita of the same year (World Bank, code `WLD`). A ratio above 1 means that the visited country is richer than the world average. We use the median and not the mean, because a few very rich countries (Luxembourg, Switzerland...) pull the mean up.

In [ ]:
# World GDP per capita each year (the World Bank gives it with the code "WLD")
world_gdp = wb[wb["iso3"] == "WLD"][["year", "gdp_per_capita"]]
world_gdp = world_gdp.rename(columns={"gdp_per_capita": "world_gdp_per_capita"})

visits = visits.merge(world_gdp, on="year", how="left")
visits["gdp_ratio"] = visits["gdp_per_capita"] / visits["world_gdp_per_capita"]

# Visits without GDP data
visits.loc[visits["gdp_per_capita"].isna(), ["pope", "year", "country"]]

Only 9 visits out of 296 have no GDP data. They are mostly the trips of John-Paul II to Poland before 1990 (the World Bank series starts after the fall of the communist regime) and very small territories (Guam, San Marino, Liechtenstein...). For Francis, only South Sudan is missing. We drop these visits: it slightly reduces the sample of John-Paul II but should not change the results.

In [ ]:
gdp = visits.dropna(subset=["gdp_ratio"]).copy()
gdp["richer_than_world"] = gdp["gdp_ratio"] > 1

gdp_table = gdp.groupby("pope").agg(
    nb_visits=("gdp_ratio", "size"),
    median_gdp_visits=("gdp_per_capita", "median"),
    median_gdp_world=("world_gdp_per_capita", "median"),
    median_ratio=("gdp_ratio", "median"),
    pct_richer_than_world=("richer_than_world", "mean"),
).loc[POPES]
gdp_table["pct_richer_than_world"] = gdp_table["pct_richer_than_world"] * 100
gdp_table.round(2)

The world GDP per capita is not the same for the three popes because it grows over time: this is why we compare the ratios and not the GDP directly.

The three popes have very different profiles. Half of the visits of John-Paul II were in countries at least 30 % poorer than the world average (median ratio of 0.70) and only 41 % of his visits were in countries richer than the world. On the opposite, Benedict XVI mostly went to rich countries: the median visited country was almost 2.5 times richer than the world and 73 % of his visits were above the world average. Francis is exactly in the middle, with a median ratio of 1 and half of his visits on each side.

This is consistent with the previous parts: Benedict XVI focused on Europe and on a few large American countries, while John-Paul II covered a large part of Africa and Latin America. However, the GDP ratio counts every visit, so the countries visited several times (Poland, France, Spain...) weigh more. We therefore also look at the income groups and at the number of different countries visited.

### Number of visits by income group

We use the World Bank income groups. Be careful: it is the current classification, not the one of the year of the visit.

In [ ]:
# Number of visits in each income group (from the poorest to the richest)
income_order = ["Low income", "Lower middle income", "Upper middle income", "High income"]
by_income = pd.crosstab(visits["income"], visits["pope"]).loc[income_order, POPES]

by_income.plot.bar(figsize=(9, 4))
plt.xticks(rotation=0)
plt.xlabel("")
plt.ylabel("Number of visits")
plt.title("Number of visits by income group of the visited country")
plt.show()
by_income

In [ ]:
# Number of different countries visited in each income group (a country visited twice counts once)
countries_by_income = visits.groupby(["income", "pope"])["iso3"].nunique().unstack(fill_value=0).loc[income_order, POPES]

# Total number of countries in each income group, to compare
countries_by_income["nb_countries"] = wb_countries["income"].value_counts()

# Share of the countries of the group that each pope visited (in %)
for pope in POPES:
    countries_by_income["% " + pope] = (countries_by_income[pope] / countries_by_income["nb_countries"] * 100).round(1)
countries_by_income

In number of visits, the high income countries come first for the three popes (about half of the visits of John-Paul II and Francis, more than two thirds for Benedict XVI). This seems to contradict the GDP ratio of John-Paul II, but it comes from the classification: it is the current one, and many countries he visited in the 1980s and 1990s are now high income (Poland, South Korea, Chile, Uruguay, Croatia...). The GDP of the year of the visit is therefore a better indicator for him.

The share of the countries of each group that were visited is more enlightening:

- **John-Paul II** visited between 56 % and 66 % of the countries of every group. He did not favour rich or poor countries: he simply went almost everywhere.
- **Benedict XVI** never went to a low income country and visited only a few lower and upper middle income countries. His trips are clearly oriented towards rich countries.
- **Francis** visited about a third of the upper middle and high income countries, but only 17 % of the lower middle income countries and 24 % of the low income ones. Despite his discourse on the "peripheries" and the "Church of the poor", his trips do not target the poorest countries more than the others. Among the six low income countries he visited, three were at war or had just come out of one (Central African Republic, South Sudan, Democratic Republic of the Congo), which suggests that peace mattered at least as much as poverty in his choices.

We should keep in mind that the high income group also contains many small territories (Caribbean islands, Gulf states, micro-states) that popes rarely visit, which lowers the share of visited countries in this group.

To complete the analysis, we could cross the income group with the share of Catholics, to see whether the popes go to poor countries only when they are Catholic, or whether poverty is a criterion on its own.